# Visium HD Spatial Transcriptomics Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stevenpastor/spatial_transcriptomics_resources/blob/main/notebook/visium_hd_quickstart.ipynb)

Condensed walkthrough of Visium HD analysis on the **10x Genomics Human CRC Patient 1** dataset ([Nature Genetics 2025](https://www.nature.com/articles/s41588-025-02193-3)). For detailed explanations, see the [comprehensive tutorial](https://colab.research.google.com/github/stevenpastor/spatial_transcriptomics_resources/blob/main/notebook/visium_hd_crc_p1_tutorial.ipynb).

---

## Setup

In [ ]:
# Package installation + data download
import os, sys, subprocess
from pathlib import Path

# Install packages
print("Installing packages...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "scanpy>=1.10", "squidpy>=1.6", "anndata>=0.10",
    "numpy>=1.24", "pandas>=2.0", "scipy>=1.11",
    "matplotlib>=3.7", "seaborn>=0.13",
    "scikit-image>=0.21", "Pillow>=10",
    "igraph", "leidenalg",
    "tqdm>=4.65", "requests>=2.31", "h5py>=3.9", "pyarrow>=13",
])
print("Done.")

# Download precomputed data from Figshare (per-file via API, no zip)
FIGSHARE_ARTICLE_ID = "31937262"
FIGSHARE_API = f"https://api.figshare.com/v2/articles/{FIGSHARE_ARTICLE_ID}/files"

DATA_DIR = Path("/content/data")
PRECOMPUTED_DIR = DATA_DIR / "precomputed"
SPATIAL_DIR = PRECOMPUTED_DIR / "spatial"
PRECOMPUTED_DIR.mkdir(parents=True, exist_ok=True)
SPATIAL_DIR.mkdir(parents=True, exist_ok=True)

RAW_H5AD = PRECOMPUTED_DIR / "crc_p1_raw_50k.h5ad"
ANNOT_H5AD = PRECOMPUTED_DIR / "crc_p1_annotated_50k.h5ad"

SPATIAL_FILENAMES = {
    "tissue_hires_image.png",
    "tissue_lowres_image.png",
    "scalefactors_json.json",
}

def _target_path(name):
    return SPATIAL_DIR / name if name in SPATIAL_FILENAMES else PRECOMPUTED_DIR / name

def download_file(url, dest, desc="Downloading"):
    """Stream a single file to disk with a progress bar and atomic rename."""
    import requests
    from tqdm.auto import tqdm
    tmp = dest.with_suffix(dest.suffix + ".part")
    with requests.get(url, stream=True, timeout=(60, None)) as resp:
        resp.raise_for_status()
        total = int(resp.headers.get("content-length", 0))
        with open(tmp, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=desc) as pbar:
            for chunk in resp.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    tmp.replace(dest)
    print(f"Saved: {dest.relative_to(DATA_DIR)} ({dest.stat().st_size / 1e6:.2f} MB)")

def fetch_figshare_files():
    import requests
    resp = requests.get(FIGSHARE_API, timeout=60)
    resp.raise_for_status()
    return resp.json()

needed = [
    RAW_H5AD,
    ANNOT_H5AD,
    SPATIAL_DIR / "tissue_hires_image.png",
    SPATIAL_DIR / "tissue_lowres_image.png",
    SPATIAL_DIR / "scalefactors_json.json",
]

if not all(p.exists() for p in needed):
    print("Downloading precomputed data from Figshare (per file)...")
    for entry in fetch_figshare_files():
        name = entry["name"]
        dest = _target_path(name)
        expected_size = entry.get("size")
        if dest.exists() and expected_size is not None and dest.stat().st_size == expected_size:
            print(f"Skipping (already present): {dest.relative_to(DATA_DIR)}")
            continue
        download_file(entry["download_url"], dest, desc=name)
    print("Download complete.")
else:
    print("Precomputed data already downloaded.")

# Clone repo for utils.py
REPO_URL = "https://github.com/stevenpastor/spatial_transcriptomics_resources.git"
REPO_DIR = Path("/content/spatial_tutorial")
if not REPO_DIR.exists():
    print("Cloning tutorial repo (for utility functions)...")
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
SCRIPTS_DIR = REPO_DIR / "scripts"
print("Ready.")

In [ ]:
import sys, os, time, warnings, gc
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False, fontsize=12)

# Paths
DATA_DIR = Path("/content/data")
PRECOMPUTED_DIR = DATA_DIR / "precomputed"
FIG_DIR = Path("/content/figures")
FIG_DIR.mkdir(exist_ok=True)

sys.path.insert(0, str(SCRIPTS_DIR))
from utils import (
    load_visium_hd_binned,
    compute_qc_metrics,
    spatial_qc_plots,
    spatial_outlier_detection,
)

print(f"scanpy {sc.__version__}, squidpy {sq.__version__}")


## Data Loading

Load pre-processed 8 um binned data (50K bins) with ground-truth annotations from the original study group.

In [ ]:
%%time
# List precomputed files
import os
for f in sorted(PRECOMPUTED_DIR.iterdir()):
    size_mb = f.stat().st_size / 1e6 if f.is_file() else 0
    print(f"  {f.name:40s}  {size_mb:6.1f} MB" if f.is_file() else f"  {f.name}/")

# Load 8 um binned data
adata_8um = sc.read_h5ad(PRECOMPUTED_DIR / "crc_p1_raw_50k.h5ad")
print(f"\nLoaded: {adata_8um.shape[0]:,} bins x {adata_8um.shape[1]:,} genes")
print(f"Spatial range X: [{adata_8um.obsm['spatial'][:,0].min():.0f}, {adata_8um.obsm['spatial'][:,0].max():.0f}]")
print(f"Spatial range Y: [{adata_8um.obsm['spatial'][:,1].min():.0f}, {adata_8um.obsm['spatial'][:,1].max():.0f}]")
if sparse.issparse(adata_8um.X):
    print(f"Median UMIs/bin: {np.median(adata_8um.X.sum(axis=1).A1):.0f}")
    print(f"Median genes/bin: {np.median((adata_8um.X > 0).sum(axis=1).A1):.0f}")

# Ground-truth annotations (from the original study group)
if "UnsupervisedL1" in adata_8um.obs.columns:
    print(f"\nGround-truth annotations present: {adata_8um.obs['UnsupervisedL1'].notna().sum():,} / {adata_8um.shape[0]:,} bins")
    print(adata_8um.obs["UnsupervisedL1"].value_counts().to_string())

## H&E Image Verification

Overlay bin positions on the H&E to verify spatial registration.

In [ ]:
%%time
# Load H&E and overlay bin positions
import json as _json

# Image is embedded in the h5ad by generate_precomputed.py
if "spatial_image" in adata_8um.uns:
    he_image = adata_8um.uns["spatial_image"]
    scalefactors = adata_8um.uns.get("scalefactors", {})
    sf = scalefactors.get("tissue_hires_scalef", scalefactors.get("tissue_lowres_scalef", 1.0))
    print(f"Image from adata.uns, shape={he_image.shape}, sf={sf}")
else:
    # Fallback: check spatial/ directory
    from skimage import io as skio
    spatial_dir = PRECOMPUTED_DIR / "spatial"
    he_image = None
    for name in ["tissue_hires_image.png", "tissue_lowres_image.png"]:
        candidate = spatial_dir / name
        if candidate.exists():
            he_image = skio.imread(str(candidate))
            with open(spatial_dir / "scalefactors_json.json") as f:
                scalefactors = _json.load(f)
            sf = scalefactors.get("tissue_hires_scalef", scalefactors.get("tissue_lowres_scalef", 1.0))
            break

if he_image is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    axes[0].imshow(he_image)
    axes[0].set_title("H&E Image")
    axes[0].axis("off")

    coords = adata_8um.obsm["spatial"]
    axes[1].imshow(he_image)
    axes[1].scatter(
        coords[:, 0] * sf, coords[:, 1] * sf,
        s=0.1, c="red", alpha=0.2, rasterized=True,
    )
    axes[1].set_title("8 µm bins overlaid on H&E")
    axes[1].axis("off")

    plt.tight_layout()
    plt.savefig(FIG_DIR / "crc_image_verification.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Bins plotted: {coords.shape[0]:,}")
else:
    print("H&E image not found. Plotting spatial coordinates only.")
    fig, ax = plt.subplots(figsize=(8, 7))
    coords = adata_8um.obsm["spatial"]
    total_counts = np.asarray(adata_8um.X.sum(axis=1)).flatten() if sparse.issparse(adata_8um.X) else adata_8um.X.sum(axis=1)
    sc_plot = ax.scatter(coords[:, 0], coords[:, 1], s=0.1,
                         c=total_counts,
                         cmap="viridis", edgecolors="none", rasterized=True)
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
    ax.set_title("Bin positions (colored by UMI count)")
    plt.colorbar(sc_plot, ax=ax, shrink=0.6)
    plt.tight_layout()
    plt.show()


### What this means

* The left panel shows the H&E image; the right overlays bin coordinates. The dot cloud should match the tissue boundary with no offset or rotation.

* Dots look sparse because this is a ~50K subsample of ~500K+ bins. That is expected.

## Quality Control

| Metric | What it measures | High values suggest |
|--------|-----------------|-------------------|
| `total_counts` | Total UMIs per bin | Dense tissue / doublets |
| `n_genes_by_counts` | Genes detected per bin | Transcriptomic complexity |
| `pct_counts_mt` | % mitochondrial UMIs | Dying/stressed cells |
| `complexity` | log(genes)/log(counts) | Transcriptomic diversity |

This CRC tissue has genuine quality variation: tissue folds, necrotic regions, edge artifacts.

In [ ]:
%%time
# --- Compute QC metrics ---
adata_8um = compute_qc_metrics(adata_8um)

print("QC metric summary:")
for col in ["total_counts", "n_genes_by_counts", "pct_counts_mt", "pct_counts_ribo", "complexity"]:
    vals = adata_8um.obs[col]
    print(f"  {col:25s}  median={vals.median():8.1f}  mean={vals.mean():8.1f}  "
          f"[{vals.min():.1f}, {vals.max():.1f}]")

# --- Violin plots ---
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, metric in zip(axes, ["total_counts", "n_genes_by_counts",
                              "pct_counts_mt", "pct_counts_ribo", "complexity"]):
    parts = ax.violinplot(adata_8um.obs[metric].dropna().values, showmedians=True, showextrema=False)
    for pc in parts["bodies"]:
        pc.set_facecolor("steelblue"); pc.set_alpha(0.7)
    ax.set_title(metric.replace("_", "\n"), fontsize=10)
    ax.set_xticks([])
plt.suptitle("QC Distributions (8 um bins)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_qc_violins.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Scatter: genes vs UMIs colored by mt% ---
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    adata_8um.obs["total_counts"], adata_8um.obs["n_genes_by_counts"],
    c=adata_8um.obs["pct_counts_mt"], s=0.5, cmap="RdYlBu_r",
    edgecolors="none", rasterized=True,
    vmin=0, vmax=adata_8um.obs["pct_counts_mt"].quantile(0.95),
)
ax.set_xlabel("Total UMI counts"); ax.set_ylabel("Genes detected")
ax.set_title("Gene complexity colored by mitochondrial %")
plt.colorbar(scatter, ax=ax, label="pct_counts_mt")
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_qc_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Histograms with thresholds ---
min_counts = max(adata_8um.obs["total_counts"].quantile(0.01), 10)
max_counts = adata_8um.obs["total_counts"].quantile(0.995)
min_genes = max(adata_8um.obs["n_genes_by_counts"].quantile(0.01), 5)
max_mt = min(adata_8um.obs["pct_counts_mt"].quantile(0.95), 25)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
thresholds = [
    ("total_counts", min_counts, max_counts, "UMI Counts"),
    ("n_genes_by_counts", min_genes, None, "Genes Detected"),
    ("pct_counts_mt", None, max_mt, "Mitochondrial %"),
    ("pct_counts_ribo", None, None, "Ribosomal %"),
    ("complexity", None, None, "Complexity"),
]
for ax, (metric, lo, hi, label) in zip(axes.flat, thresholds):
    ax.hist(adata_8um.obs[metric].dropna(), bins=100, color="steelblue", alpha=0.7, edgecolor="none")
    if lo is not None:
        ax.axvline(lo, color="red", ls="--", lw=1.5, label=f"min={lo:.1f}")
    if hi is not None:
        ax.axvline(hi, color="red", ls="--", lw=1.5, label=f"max={hi:.1f}")
    ax.set_title(label)
    if lo is not None or hi is not None:
        ax.legend(fontsize=8)
axes[1, 2].axis("off")
plt.suptitle("QC Histograms with Suggested Thresholds", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_qc_histograms.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Suggested thresholds: min_counts={min_counts:.1f}, max_counts={max_counts:.1f}, min_genes={min_genes:.1f}, max_mt={max_mt:.1f}%")

# --- Spatial QC heatmaps ---
spatial_qc_plots(
    adata_8um,
    metrics=["total_counts", "n_genes_by_counts", "pct_counts_mt", "pct_counts_ribo", "complexity"],
    figsize=(25, 4), spot_size=0.3,
    save=str(FIG_DIR / "crc_spatial_qc_heatmaps.png"),
)

# --- Spatial outlier detection ---
print("Computing spatial neighbors (k=20)...")
sq.gr.spatial_neighbors(adata_8um, n_neighs=20, coord_type="generic")

outlier_metrics = ["total_counts", "pct_counts_mt", "n_genes_by_counts"]
total_outliers = np.zeros(adata_8um.shape[0], dtype=bool)
for metric in outlier_metrics:
    outliers = spatial_outlier_detection(adata_8um, metric=metric, z_threshold=3.0)
    n_out = outliers.sum()
    print(f"  {metric:25s}: {n_out:,} outliers ({100*n_out/len(outliers):.2f}%)")
    total_outliers = total_outliers | outliers.values

adata_8um.obs["spatial_outlier"] = total_outliers
print(f"\nTotal spatial outliers: {total_outliers.sum():,} ({100*total_outliers.mean():.2f}%)")

fig, ax = plt.subplots(figsize=(8, 7))
coords = adata_8um.obsm["spatial"]
ax.scatter(coords[~total_outliers, 0], coords[~total_outliers, 1],
           s=0.1, c="lightgray", alpha=0.3, rasterized=True, label="Normal")
ax.scatter(coords[total_outliers, 0], coords[total_outliers, 1],
           s=1, c="red", alpha=0.6, rasterized=True, label="Outlier")
ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
ax.legend(markerscale=10, fontsize=10)
ax.set_title(f"Spatial outliers (n={total_outliers.sum():,})")
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_spatial_outliers.png", dpi=150, bbox_inches="tight")
plt.show()

### What this means

* **Per-bin metrics**: Median ~147 UMIs and ~122 genes per bin. Expected for 8 um resolution. Mitochondrial % has a median of 6.7% but a long tail to 100% (dead/damaged bins). Ribosomal % is zero because the 10x probe panel excludes RPL*/RPS* genes.

* **Scatter plot**: High-mt bins (red/yellow) fall below the diagonal, meaning high counts but few unique genes. Classic damage signature. Supports using both mt% and gene count filters together.

* **Spatial QC**: The heatmaps show a gradient with higher signal in the tumor region and lower signal in the stroma. No localized artifacts. Spatial outlier detection flagged ~4% of bins that deviate from their local neighborhood, scattered across the tissue.

* **Why spatial QC matters**: A bin with 50 UMIs in dense tumor is suspicious; the same bin in sparse stroma is normal. Local outlier detection catches bins that do not match their surroundings.

## Filtering and Normalization

In [ ]:
%%time
# --- Apply QC filters ---
n_before = adata_8um.shape[0]

min_counts = max(adata_8um.obs["total_counts"].quantile(0.01), 10)
max_counts = adata_8um.obs["total_counts"].quantile(0.995)
min_genes = max(adata_8um.obs["n_genes_by_counts"].quantile(0.01), 5)
max_mt = min(adata_8um.obs["pct_counts_mt"].quantile(0.95), 25)

keep = (
    (adata_8um.obs["total_counts"] >= min_counts) &
    (adata_8um.obs["total_counts"] <= max_counts) &
    (adata_8um.obs["n_genes_by_counts"] >= min_genes) &
    (adata_8um.obs["pct_counts_mt"] <= max_mt) &
    (~adata_8um.obs["spatial_outlier"].astype(bool))
)
adata_8um = adata_8um[keep].copy()

min_cells = max(int(0.001 * adata_8um.shape[0]), 3)
sc.pp.filter_genes(adata_8um, min_cells=min_cells)

print(f"Filtering: {n_before:,} -> {adata_8um.shape[0]:,} bins "
      f"(removed {n_before - adata_8um.shape[0]:,}, "
      f"{100*(n_before - adata_8um.shape[0])/n_before:.1f}%)")
print(f"Genes: {adata_8um.shape[1]:,}")

# --- Load pre-annotated data (normalized + PCA + UMAP + Leiden) ---
# I precomputed this object with normalization, PCA, UMAP, and Leiden clustering.
# Bin count differs slightly from the filtering above because I ran it separately.
adata_8um = sc.read_h5ad(PRECOMPUTED_DIR / "crc_p1_annotated_50k.h5ad")
print(f"\nLoaded annotated data: {adata_8um.shape}")
print(f"Leiden clusters: {adata_8um.obs['leiden'].nunique()}")

# --- UMAP colored by QC metrics + Leiden ---
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, metric, cmap in zip(axes[:3],
    ["total_counts", "n_genes_by_counts", "pct_counts_mt"],
    ["viridis", "viridis", "RdYlBu_r"]):
    sc_p = ax.scatter(adata_8um.obsm["X_umap"][:, 0], adata_8um.obsm["X_umap"][:, 1],
                      c=adata_8um.obs[metric], s=0.3, cmap=cmap,
                      edgecolors="none", rasterized=True)
    ax.set_title(metric); plt.colorbar(sc_p, ax=ax, shrink=0.7)

for cl in sorted(adata_8um.obs["leiden"].unique(), key=int):
    mask = adata_8um.obs["leiden"] == cl
    axes[3].scatter(adata_8um.obsm["X_umap"][mask, 0], adata_8um.obsm["X_umap"][mask, 1],
                    s=0.3, label=cl, alpha=0.5, rasterized=True)
axes[3].set_title("Leiden clusters")
axes[3].legend(markerscale=5, fontsize=7, ncol=2, loc="best")
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_umap_qc_leiden.png", dpi=150, bbox_inches="tight")
plt.show()

### What this means

* Combined spatial and global filtering removed ~22% of bins. The remaining bins show clear UMAP structure.

* High-count/high-mt bins cluster together in the bottom-right of the UMAP. Whether this reflects biology (active tumor epithelium) or QC artifacts is resolved by mapping spatially, which we do next.

## Ground-Truth Cell Type Annotations

These annotations come from the original study group ([de Oliveira et al., Nature Genetics 2025](https://www.nature.com/articles/s41588-025-02193-3)), not from us. We use them as our reference.

Two levels: **UnsupervisedL1** (10 broad categories) and **DeconvolutionLabel1** (38 detailed types).

In [ ]:
# --- Visualize ground-truth UnsupervisedL1 ---
ct_col = "UnsupervisedL1"
has_label = adata_8um.obs[ct_col].notna()
adata_labeled = adata_8um[has_label].copy()
print(f"Bins with {ct_col}: {has_label.sum():,} / {adata_8um.shape[0]:,}")

cell_types = sorted(adata_labeled.obs[ct_col].dropna().unique())
palette = dict(zip(cell_types, plt.cm.tab20(np.linspace(0, 1, len(cell_types)))))

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ct in cell_types:
    mask = (adata_labeled.obs[ct_col] == ct).values
    axes[0].scatter(adata_labeled.obsm["X_umap"][mask, 0], adata_labeled.obsm["X_umap"][mask, 1],
                    s=0.5, label=ct, alpha=0.5, rasterized=True, color=palette[ct])
axes[0].legend(markerscale=8, fontsize=8, loc="best", framealpha=0.8)
axes[0].set_title(f"UMAP: {ct_col} (ground truth)"); axes[0].set_xlabel("UMAP 1"); axes[0].set_ylabel("UMAP 2")

coords = adata_labeled.obsm["spatial"]
for ct in cell_types:
    mask = (adata_labeled.obs[ct_col] == ct).values
    axes[1].scatter(coords[mask, 0], coords[mask, 1],
                    s=0.3, label=ct, alpha=0.4, rasterized=True, color=palette[ct])
axes[1].legend(markerscale=10, fontsize=8, loc="best", framealpha=0.8)
axes[1].set_title(f"Spatial: {ct_col}"); axes[1].set_aspect("equal"); axes[1].invert_yaxis(); axes[1].axis("off")

plt.tight_layout()
plt.savefig(FIG_DIR / "crc_ground_truth_unsupervised.png", dpi=150, bbox_inches="tight")
plt.show()

# Proportions
fig, ax = plt.subplots(figsize=(10, 5))
props = adata_labeled.obs[ct_col].value_counts(normalize=True).sort_values(ascending=True)
colors = [palette.get(ct, "gray") for ct in props.index]
ax.barh(range(len(props)), props.values, color=colors)
ax.set_yticks(range(len(props))); ax.set_yticklabels(props.index)
ax.set_xlabel("Proportion"); ax.set_title(f"{ct_col} proportions")
for i, v in enumerate(props.values):
    ax.text(v + 0.005, i, f"{v:.1%}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_ground_truth_proportions.png", dpi=150, bbox_inches="tight")
plt.show()

### What this means

* **UMAP**: Cell types separate well. Tumor forms a large cluster in the bottom-center/right, fibroblast sits adjacent, intestinal epithelial occupies the left. The high-count/high-mt clusters correspond to tumor and epithelial compartments, so the elevated mt% there is likely biological.

* **Spatial**: Cell types map to anatomically sensible regions. Tumor occupies a contiguous zone, fibroblasts surround it, epithelium lines the mucosal surface, immune populations are interspersed.

* **Proportions**: Tumor and Unknown tied at ~21.5% each. The high Unknown fraction reflects ambiguous bins where multiple cells overlap at 8 um.

## Our Marker-Based Annotations

Now we do our own annotation using canonical CRC marker gene sets, then compare against the ground truth from the original study.

In [ ]:
%%time
# --- Marker-based annotation ---
print("Scoring marker panels...")

marker_sets = {
    "Tumor_Epithelial": ["EPCAM", "KRT8", "KRT18", "KRT19", "KRT20", "CEACAM5", "MUC1"],
    "Fibroblast":       ["COL1A1", "COL1A2", "COL3A1", "DCN", "LUM", "COL6A1"],
    "Endothelial":      ["PECAM1", "VWF", "EMCN", "KDR", "RAMP2", "PLVAP"],
    "Myeloid":          ["LST1", "TYROBP", "FCER1G", "CTSS", "AIF1", "C1QC", "C1QB"],
    "T_NK":             ["CD3D", "CD3E", "TRBC1", "TRBC2", "NKG7", "IL7R", "LTB"],
    "B_Plasma":         ["MS4A1", "CD79A", "CD79B", "MZB1", "JCHAIN", "IGKC", "CD74"],
    "Smooth_Muscle":    ["ACTA2", "MYH11", "DES", "TAGLN", "CNN1", "ACTG2"],
    "Goblet":           ["MUC2", "TFF3", "FCGBP", "CLCA1", "ZG16", "AGR2"],
    "Enterocyte":       ["FABP1", "FABP2", "ALPI", "VIL1", "SIS"],
    "Mast":             ["TPSAB1", "TPSB2", "KIT", "CPA3", "HDC"],
}

present_markers = {}
for ct, genes in marker_sets.items():
    present = [g for g in genes if g in adata_8um.var_names]
    present_markers[ct] = present
    print(f"  {ct:20s}  {len(present):>2}/{len(genes)} genes  {present}")

score_cols = []
for ct, genes in present_markers.items():
    if len(genes) >= 2:
        score_col = f"{ct}_score"
        sc.tl.score_genes(adata_8um, gene_list=genes, score_name=score_col, use_raw=False)
        score_cols.append(score_col)
    else:
        print(f"  WARNING: {ct} has <2 markers, skipping")

score_df = adata_8um.obs[score_cols].copy()
adata_8um.obs["cell_type_marker"] = score_df.idxmax(axis=1).str.replace("_score", "", regex=False)

max_score = score_df.max(axis=1)
sorted_scores = np.sort(score_df.values, axis=1)
second_score = sorted_scores[:, -2] if sorted_scores.shape[1] > 1 else np.zeros(len(score_df))
margin = max_score.values - second_score
adata_8um.obs.loc[(max_score < 0.05) | (margin < 0.02), "cell_type_marker"] = "Low_confidence"

print("\nMarker-based cell type distribution:")
print(adata_8um.obs["cell_type_marker"].value_counts().to_string())

# --- Marker score heatmap ---
score_summary = (
    adata_8um.obs.groupby("cell_type_marker")[score_cols]
    .median()
    .rename(columns=lambda x: x.replace("_score", ""))
)
fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(score_summary.values, aspect="auto", cmap="viridis")
ax.set_xticks(np.arange(score_summary.shape[1]))
ax.set_xticklabels(score_summary.columns, rotation=45, ha="right")
ax.set_yticks(np.arange(score_summary.shape[0]))
ax.set_yticklabels(score_summary.index)
ax.set_title("Median marker scores by assigned cell type")
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_marker_score_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Compare against ground truth ---
marker_to_gt = {
    "Tumor_Epithelial": "Tumor",
    "Fibroblast":       "Fibroblast",
    "Endothelial":      "Endothelial",
    "Myeloid":          "Myeloid",
    "T_NK":             "T cells",
    "B_Plasma":         "B cells",
    "Smooth_Muscle":    "Smooth Muscle",
    "Goblet":           "Intestinal Epithelial",
    "Enterocyte":       "Intestinal Epithelial",
    "Mast":             "Myeloid",
}

adata_8um.obs["marker_mapped"] = adata_8um.obs["cell_type_marker"].map(marker_to_gt).fillna("Other")

has_both = (adata_8um.obs["UnsupervisedL1"].notna()) & (adata_8um.obs["cell_type_marker"] != "Low_confidence")
if has_both.sum() > 0:
    agree = (adata_8um.obs.loc[has_both, "marker_mapped"] == adata_8um.obs.loc[has_both, "UnsupervisedL1"])
    print(f"\nAgreement (marker vs ground-truth): {agree.mean():.1%}  ({has_both.sum():,} bins)")

    cm = pd.crosstab(
        adata_8um.obs.loc[has_both, "marker_mapped"],
        adata_8um.obs.loc[has_both, "UnsupervisedL1"],
        normalize="index",
    )
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", ax=ax)
    ax.set_xlabel("Ground Truth (UnsupervisedL1)")
    ax.set_ylabel("Marker-Based (mapped)")
    ax.set_title(f"Annotation Agreement: {agree.mean():.1%}")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "crc_annotation_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

# --- Spatial map of marker annotations ---
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
ct_col = "cell_type_marker"
cell_types = sorted(adata_8um.obs[ct_col].unique())
palette = dict(zip(cell_types, plt.cm.tab20(np.linspace(0, 1, len(cell_types)))))

for ct in cell_types:
    mask = (adata_8um.obs[ct_col] == ct).values
    axes[0].scatter(adata_8um.obsm["X_umap"][mask, 0], adata_8um.obsm["X_umap"][mask, 1],
                    s=0.5, label=ct, alpha=0.5, rasterized=True, color=palette[ct])
axes[0].legend(markerscale=8, fontsize=7, loc="best"); axes[0].set_title("UMAP: Marker-based")

coords = adata_8um.obsm["spatial"]
for ct in cell_types:
    mask = (adata_8um.obs[ct_col] == ct).values
    axes[1].scatter(coords[mask, 0], coords[mask, 1],
                    s=0.3, label=ct, alpha=0.4, rasterized=True, color=palette[ct])
axes[1].legend(markerscale=10, fontsize=7, loc="best")
axes[1].set_title("Spatial: Marker-based"); axes[1].set_aspect("equal"); axes[1].invert_yaxis(); axes[1].axis("off")
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_marker_cell_type_maps.png", dpi=150, bbox_inches="tight")
plt.show()

### What this means

* **Scoring**: Each bin gets assigned to the highest-scoring marker panel. Low-confidence bins (max score <0.05 or margin <0.02 over second-best) are flagged separately.

* **Heatmap**: Most cell types show their highest score on the matching panel (bright diagonal). Fibroblast and Tumor_Epithelial cross-react at the tumor-stroma interface.

* **Agreement with ground truth: ~55%**. Tumor (0.80) and T cells (0.84) do well because their markers are specific. Fibroblast (0.42) and B cells (0.17) are weak because their markers (COL1A1, CD74) are broadly expressed.

* **Spatial patterns**: Despite moderate quantitative agreement, the marker annotations produce broadly similar spatial compartments as the ground truth. The disagreements are in cell types with non-specific markers, not in spatial logic.

## Neighborhood Analysis

Which cell types spatially co-localize? In CRC we expect tumor to self-cluster, fibroblasts (CAFs) to surround tumor, immune cells in the periphery, and intestinal epithelium in normal tissue regions.

In [ ]:
%%time
# --- Neighborhood enrichment ---
ct_key = "UnsupervisedL1"
adata_nhood = adata_8um[adata_8um.obs[ct_key].notna()].copy()
adata_nhood.obs[ct_key] = adata_nhood.obs[ct_key].astype("category")

sq.gr.spatial_neighbors(adata_nhood, n_neighs=20, coord_type="generic")
sq.gr.nhood_enrichment(adata_nhood, cluster_key=ct_key)

fig, ax = plt.subplots(figsize=(8, 7))
sq.pl.nhood_enrichment(adata_nhood, cluster_key=ct_key, ax=ax)
ax.set_title("Neighborhood Enrichment (UnsupervisedL1)")
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_nhood_enrichment.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Co-occurrence analysis ---
try:
    sq.gr.co_occurrence(adata_nhood, cluster_key=ct_key)

    co = adata_nhood.uns[f"{ct_key}_co_occurrence"]
    occ = co["occ"]
    interval = co["interval"]

    if len(interval) == occ.shape[-1] + 1:
        x = (interval[:-1] + interval[1:]) / 2
    else:
        x = interval

    cats = list(adata_nhood.obs[ct_key].cat.categories)
    n = len(cats)
    ncols = 5
    nrows = int(np.ceil(n / ncols))
    palette_co = dict(zip(cats, plt.cm.tab10(np.linspace(0, 1, n))))

    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows),
                             sharex=True, sharey=True)
    axes_flat = axes.flat if hasattr(axes, "flat") else [axes]

    for i, anchor in enumerate(cats):
        ax = axes_flat[i]
        for j, other in enumerate(cats):
            ax.plot(x, occ[i, j, :], label=other, color=palette_co[other], lw=1.2)
        ax.axhline(1.0, color="gray", ls="--", lw=0.7)
        ax.set_title(f"Anchor: {anchor}", fontsize=9)
        ax.set_xlabel("Distance"); ax.set_ylabel("P(other | anchor)")

    for k in range(n, len(list(axes_flat))):
        axes_flat[k].axis("off")

    handles, labels = axes_flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="center right", fontsize=8,
               bbox_to_anchor=(1.02, 0.5), title=ct_key)
    plt.suptitle("Co-occurrence by cell type", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "crc_co_occurrence.png", dpi=150, bbox_inches="tight")
    plt.show()
except Exception as e:
    print(f"Co-occurrence analysis: {e}")

del adata_nhood; gc.collect()

### What this means

* **Neighborhood enrichment**: Tests whether cell type pairs co-localize more or less than random permutation expects. Strong self-enrichment along the diagonal for Tumor, Smooth Muscle, and Intestinal Epithelial. Tumor avoids most other types, consistent with immune exclusion.

* **Co-occurrence**: Shows how co-localization changes with distance. Tumor is self-enriched at short range with all other types below 1 (avoidance). Neuronal bins massively self-enrich at short range and co-localize with Smooth Muscle, consistent with the muscularis layer.

* **Key finding**: The tissue has well-defined spatial compartmentalization. The immune exclusion pattern from the tumor mass is the most translationally relevant observation.

## Spatially Variable Genes

Identify genes with significant spatial structure using Moran's I autocorrelation. Subsampled to 10K bins for speed.

In [ ]:
%%time
# --- SVG detection ---
n_svg = min(10_000, adata_8um.shape[0])
print(f"Subsampling to {n_svg:,} bins for SVG analysis...")
adata_svg = sc.pp.subsample(adata_8um, n_obs=n_svg, copy=True)
sq.gr.spatial_neighbors(adata_svg, n_neighs=20, coord_type="generic")

if "highly_variable" in adata_svg.var.columns:
    adata_svg_hvg = adata_svg[:, adata_svg.var["highly_variable"]].copy()
else:
    adata_svg_hvg = adata_svg.copy()

print(f"Running Moran's I on {adata_svg_hvg.shape[1]} genes...")
sq.gr.spatial_autocorr(adata_svg_hvg, mode="moran", n_jobs=1)

morans = adata_svg_hvg.uns["moranI"].sort_values("I", ascending=False)
print(f"\nTop 20 SVGs:")
print(morans.head(20)[["I", "pval_norm"]].to_string())

# --- Spatial maps of top SVGs ---
top_svgs = morans.head(8).index.tolist()
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
coords = adata_svg.obsm["spatial"]

for ax, gene in zip(axes.flat, top_svgs):
    if gene in adata_svg.var_names:
        vals = adata_svg[:, gene].X
        expr = vals.toarray().flatten() if sparse.issparse(vals) else np.asarray(vals).flatten()
    else:
        expr = np.zeros(adata_svg.shape[0])
    sc_p = ax.scatter(coords[:, 0], coords[:, 1], c=expr, s=1, cmap="magma",
                      edgecolors="none", rasterized=True)
    ax.set_title(f"{gene}\n(I={morans.loc[gene, 'I']:.3f})", fontsize=10)
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
    plt.colorbar(sc_p, ax=ax, shrink=0.6)

plt.suptitle("Top Spatially Variable Genes", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / "crc_svgs.png", dpi=150, bbox_inches="tight")
plt.show()

del adata_svg, adata_svg_hvg; gc.collect()

### What this means

* Top SVGs are biologically expected: PIGR/MUC2/FCGBP are mucosal epithelium markers, DES/MYH11 are smooth muscle, COL1A1/COL3A1 are fibroblast/stroma, CEACAM5 is tumor epithelial. They are spatially structured because the tissue compartments they mark are.

* Signal is sparse (most bins are dark) because dropout is high at 8 um depth. The spatial coherence of the non-zero bins is what Moran's I captures.

## Cell-Cell Communication

Identify ligand-receptor interactions between spatially proximal cell types.

In [ ]:
%%time
# --- L-R analysis (memory-constrained for Colab) ---
ct_key = "UnsupervisedL1"
adata_lr_src = adata_8um[adata_8um.obs[ct_key].notna()].copy()

top_cts = adata_lr_src.obs[ct_key].value_counts().head(6).index.tolist()
print(f"Restricting L-R analysis to top 6 cell types: {top_cts}")
adata_lr_src = adata_lr_src[adata_lr_src.obs[ct_key].isin(top_cts)].copy()

n_lr = min(5_000, adata_lr_src.shape[0])
adata_lr = sc.pp.subsample(adata_lr_src, n_obs=n_lr, copy=True)
del adata_lr_src
adata_lr.obs[ct_key] = adata_lr.obs[ct_key].astype("category")

sq.gr.spatial_neighbors(adata_lr, n_neighs=20, coord_type="generic")

print(f"Running L-R analysis on {adata_lr.shape[0]:,} bins, 50 permutations...")
sq.gr.ligrec(adata_lr, cluster_key=ct_key, n_perms=50, use_raw=False)
print("Done.")

# --- Plot significant L-R pairs ---
try:
    sq.pl.ligrec(adata_lr, cluster_key=ct_key,
                 pvalue_threshold=0.05, means_range=(0.3, np.inf))
    plt.savefig(FIG_DIR / "crc_ligrec.png", dpi=150, bbox_inches="tight")
    plt.show()
except Exception as e:
    print(f"L-R plot: {e}")
    print("Try adjusting pvalue_threshold or means_range if no pairs pass filters.")

del adata_lr; gc.collect()

### What this means

* L-R analysis tests whether specific ligand-receptor pairs are expressed at higher levels between spatially adjacent cell types than expected by random permutation. Significant pairs suggest active signaling.

* Look for biologically expected interactions: collagen-integrin between fibroblasts and tumor (ECM remodeling), chemokine signaling between immune and tumor cells. This is a quick screen (5K bins, 50 permutations); increase permutations for publication-quality results.

## Marker Validation

Validate annotations using a dotplot of canonical markers and visualize the Periphery labels (Tumor / Tissue / 50 um border).

In [ ]:
# --- Dotplot of canonical markers by ground-truth cell type ---
validation = {
    "Tumor":     ["EPCAM", "KRT20", "KRT8"],
    "Fibroblast": ["COL1A1", "DCN", "LUM"],
    "T cells":   ["CD3D", "CD3E"],
    "B cells":   ["MS4A1", "CD79A"],
    "Myeloid":   ["TYROBP", "C1QC", "AIF1"],
    "Endothelial": ["PECAM1", "VWF"],
    "Smooth Muscle": ["ACTA2", "MYH11"],
    "Intestinal Epithelial": ["MUC2", "TFF3", "FABP1"],
}

valid = {}
for ct, genes in validation.items():
    present = [g for g in genes if g in adata_8um.var_names]
    if present:
        valid[ct] = present

adata_gt = adata_8um[adata_8um.obs["UnsupervisedL1"].notna()].copy()
if valid:
    try:
        sc.pl.dotplot(adata_gt, var_names=valid, groupby="UnsupervisedL1",
                      use_raw=False, standard_scale="var",
                      title="Canonical markers by ground-truth cell type")
        plt.savefig(FIG_DIR / "crc_marker_dotplot.png", dpi=150, bbox_inches="tight")
        plt.show()
    except Exception as e:
        print(f"Dotplot: {e}")
        flat = [g for gs in valid.values() for g in gs]
        sc.pl.dotplot(adata_gt, var_names=flat, groupby="UnsupervisedL1", use_raw=False)
        plt.show()

# --- Periphery labels ---
if "Periphery" in adata_8um.obs.columns and adata_8um.obs["Periphery"].notna().any():
    fig, ax = plt.subplots(figsize=(9, 8))
    coords = adata_8um.obsm["spatial"]
    periph_colors = {"Tumor": "red", "Tissue": "steelblue", "50 micron": "gold"}

    for label, color in periph_colors.items():
        mask = (adata_8um.obs["Periphery"] == label).values
        if mask.any():
            ax.scatter(coords[mask, 0], coords[mask, 1], s=0.3, c=color,
                       label=f"{label} ({mask.sum():,})", alpha=0.4, rasterized=True)

    ax.legend(markerscale=10, fontsize=10)
    ax.set_title("Tumor Periphery Classification")
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "crc_periphery.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Periphery labels not available.")

### What this means

* **Dotplot**: Each marker group shows strongest expression in the expected cell type. Fibroblast markers (COL1A1/DCN/LUM) are broadly expressed across many types, confirming why our marker-based annotation over-called fibroblasts. Immune markers show small dots even in correct cell types, confirming dropout at 8 um depth.

* **Periphery**: Three zones are visible: Tumor core, normal Tissue, and a 50 um border strip tracing the invasive front. The border region is the most interesting zone for studying immune infiltration and tumor-stroma crosstalk.

## Summary

1. **QC**: Removed ~22% of bins using both global thresholds and spatial outlier detection.
2. **Ground truth**: The original study's annotations show well-separated cell types with anatomically sensible spatial patterns.
3. **Our annotations**: Marker-based scoring achieved 55% agreement with ground truth. Works well for specific markers (tumor, T cells), struggles with broadly expressed markers (fibroblast, B cells).
4. **Spatial analysis**: Neighborhood enrichment and co-occurrence confirm tumor self-clustering with immune exclusion. Top SVGs match expected tissue compartment markers.
5. **Cell-cell communication**: L-R analysis identifies signaling between spatially adjacent cell types.

For detailed explanations of each step, see the [comprehensive tutorial](https://colab.research.google.com/github/stevenpastor/spatial_transcriptomics_resources/blob/main/notebook/visium_hd_crc_p1_tutorial.ipynb).